# ScreamingFace — Platform Mode

**Platform mode** uses a ScreamingFace-hosted engine at `fusion.dev.screamingface.ai`.
No local install of services is required, and shared OpenRouter credits are available
so you do not need to provide your own API key.

### Before you start — authentication

Platform mode uses **Cloudflare Access with Google sign-in**. A login link will appear
in step 3; click it and sign in with your Google account.

> **`.gov` and institutional email note:** Cloudflare Access requires a Google account.
> If your institutional email is not a Google Workspace account (common for `.gov`
> addresses), you may not be able to log in this way. In that case, use the companion
> **BYOK notebook** (`ScreamingFace_BYOK_Guide.ipynb`) instead — it runs a local engine
> inside this Colab VM and uses your own provider API keys with no Google auth required.

### What the hosted engine provides

- Benchmarks pre-downloaded and ready (IF-Eval, DRACO, HealthBench, and others)
- Shared OpenRouter credits — models from all major providers available without your own key
- A public leaderboard you can submit results to
- **The same API surface as BYOK mode** — code written against the platform engine runs
  unchanged against a local engine by changing one `sf.configure(...)` call

### Model availability note

The hosted engine routes through OpenRouter. The full OpenRouter catalog is available,
including models from all providers. If you need to restrict to specific providers or
avoid particular model families, the BYOK path gives you full control.

---
## 1 · Install the client

In [ ]:
%pip install -q screamingface

## 2 · Import

With no arguments, `sf.Client()` defaults to the hosted engine and public leaderboard.
Displaying it shows where it points — no network calls, no cost.

In [ ]:
import screamingface as sf

sf.Client()

## 3 · Sign in

`sf.connect()` opens the connection panel. On the hosted engine a login link appears —
click it, sign in with your Google account, then return here.

Once signed in, the shared model credits are available and the panel shows which
providers are active.

In [ ]:
sf.connect()

## 4 · Explore the catalog

List available models and benchmarks on the hosted engine.

In [ ]:
sf.models.list()

In [ ]:
sf.leaderboards.list()

---
## 5 · Build candidates

The same `sf.Model`, `sf.Fusion`, and `sf.Pipeline` API as in the BYOK notebook.
The only difference is which engine executes them.

In [ ]:
IFEVAL_PARAMS = {"max_tokens": 8192, "temperature": 0.0}

ANSWER_PROMPT = (
    "Answer the request accurately and completely. "
    "Follow every instruction and formatting constraint in the request."
)

# Solo baseline
solo = sf.Model(
    model="openrouter/openai/gpt-5.5",
    prompt=ANSWER_PROMPT,
    params=IFEVAL_PARAMS,
)

solo

In [ ]:
SYNTHESIS_PROMPT = (
    "Produce the single best answer to the original request by combining the panel drafts. "
    "Preserve every instruction and formatting constraint stated in the original prompt. "
    "If drafts disagree on a constraint, follow the one that satisfies the constraint."
)

fusion = sf.Fusion(
    members=[
        sf.Model(model="openrouter/openai/gpt-5.5", prompt=ANSWER_PROMPT, params=IFEVAL_PARAMS),
        sf.Model(model="openrouter/google/gemini-3.1-flash", prompt=ANSWER_PROMPT, params=IFEVAL_PARAMS),
    ],
    name="gpt_plus_gemini",
    synthesizer=sf.Model(
        model="openrouter/google/gemini-3.1-flash",
        prompt=SYNTHESIS_PROMPT,
        params=IFEVAL_PARAMS,
    ),
)

fusion

---
## 6 · Evaluate

`limit=1` runs a single case. Remove it for the full 541-case run.
Results may be cached on the hosted engine from previous runs of the same recipe —
a full cached run costs nothing.

In [ ]:
report = sf.evaluate(
    [solo, fusion],
    benchmark="ifeval",
    limit=1,
    progress=True,
)

report

In [ ]:
for name, result in report.candidates.items():
    u = result.usage
    print(
        f"{name:30s}  score={result.score}  "
        f"cost=${u.cost_usd}  "
        f"input={u.input_tokens}tok  cache_read={u.cache_read_tokens}tok"
    )

---
## 7 · Submit to the leaderboard (optional)

Submitting publishes your fusion's score and url4 to the public leaderboard.
The url4 encodes the full recipe so the result is independently reproducible.

In [ ]:
PUBLISH = False

fusion_result = report.candidates["gpt_plus_gemini"]
print("url4:", fusion_result.url4)

submission = sf.leaderboards.submit(fusion_result) if PUBLISH else None
submission or "Set PUBLISH = True to submit."

In [ ]:
sf.leaderboards.get("ifeval", top=10)

---
## Switching to a local engine

All code above runs unchanged against a local engine. The only change is adding
`sf.configure(...)` before `sf.connect(...)` and providing your own API key:

```python
sf.configure(
    engine_url="http://127.0.0.1:9108",
    scoreboard_url="http://127.0.0.1:9106",
)
sf.connect("openrouter", api_key="sk-or-...")
```

See `ScreamingFace_BYOK_Guide.ipynb` for the full local engine setup, direct provider
options (OpenAI, Anthropic, Hugging Face), and in-depth coverage of the IF-Eval
implementation and caching architecture.

In [ ]:
sf.close()